# Location derivation: demo on the running example

Concept: `overview/Deriving Locations.md` (outside this repo).
Code: package `ocelescope_module_location_based_ocpm.derivation`; this notebook only imports and calls its functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(add_loc_2)`) and `docs/resources/location-derivation.md`.

What the notebook does:
1. loads the running example and its configuration,
2. resolves the inputs into one frame of hu-event relations,
3. checks the requirements RQ1-RQ3,
4. derives $loc_0$ to $loc_3$ step by step, then $before_{loc_k}$ and $ploc_k$,
5. runs the same with one call (`derive`),
6. saves the mapping of every level to `data/running_example/derived/`, where other notebooks load it from.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/location_derivation.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$.

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.agreement import event_locations
from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.inputs import resolve_inputs
from ocelescope_module_location_based_ocpm.derivation.level_files import (
    level_mapping,
    load_level_mapping,
    save_level_mappings,
)
from ocelescope_module_location_based_ocpm.derivation.levels import (
    add_before,
    add_loc_0,
    add_loc_1,
    add_loc_2,
    add_loc_3,
    add_ploc,
    first_location,
)
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.derivation.requirements import check_requirements

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"

# Show whole frames (the running example has 28 relations).
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250);

## 1. Log and configuration

The running example: 17 events, 6 handling units (slab s1, coils c1 to c5); time $t$ is 2026-01-01 00:00 plus $t$ hours.

The configuration is a `DerivationConfig` with one option per input (`config.py`):

| input | parameter | options (class: parameters) | running example |
|---|---|---|---|
| handling units $HU$ (required) | `handling_units` | `ObjectTypes`: `object_types`; `ObjectTable`: `table`, `object_column` | object types slab, coil |
| movement events $M$ (required) | `movement_events` | `Activities`: `activities`; `AttributeValues`: `attribute`, `values`; `EventTable`: `table`, `event_column` | crane move, load truck, unload truck |
| transformation events $T$ (optional) | `transformation_events` | as for $M$, plus `Qualifiers`: `qualifiers` (events with an e2o relation carrying one of them); `None` means $T = \emptyset$ | cast slab, roll, cut |
| target locations $uloc$ (required) | `target_locations` | `LocationAttribute`: `attribute`; `LocationQualifiers`: `qualifiers`, `name_attribute` (the related object is the location; its id or the value of `name_attribute`); `LocationTable`: `table`, `event_column`, `location_column` | event attribute `to_loc` |
| pickup locations $puloc$ (optional) | `pickup_locations` | as for $uloc$; `None` means no pickup locations | event attribute `from_loc` |
| selected level $k$ | `level` | 0, 1, 2, 3 (default 3) | 3 |

Table options refer to a table by name; the tables themselves are passed separately (parameter `tables` below), e.g. the CSV files in `data/running_example/inputs/`.
`level` does not change what is derived (all levels are always computed); it only decides which level `Derivation.loc` and the flags use.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
config

## 2. Resolve the inputs: `resolve_inputs(ocel, config, tables=None)`

Turns the configuration into the sets $HU$, $M$, $T$ and the partial functions $uloc$, $puloc$, stored as columns.

| parameter | meaning |
|---|---|
| `ocel` | the log |
| `config` | the `DerivationConfig` |
| `tables` | dict table name → polars frame; only needed for table options (default none) |

How it works: each input is resolved by its own function (`resolve_handling_units`, `resolve_event_set` for $M$ and $T$, `resolve_locations` for $uloc$ and $puloc$).
Then all events get the columns `is_movement`, `is_transformation`, `uloc`, `puloc`, and the e2o relations restricted to handling units are joined with them.
Rules applied on the way: empty values (null, NaN, `""`, `"NA"`) are never locations; table rows with unknown event ids are dropped and reported; pickup locations of non-movement events are ignored and reported; an event with two different locations in a table is a blocking error.

Returns a `ResolvedInputs` with
- `relations`: one row per hu-event relation (columns event, hu, object_type, time, activity, is_movement, is_transformation, uloc, puloc), sorted by hu, time, event;
- `events`: the same event columns for all events (also events without handling units);
- `handling_units`: all handling units with their object type;
- `notices` (non-blocking reports) and `errors` (blocking errors).

In [ ]:
resolved = resolve_inputs(ocel, config)
resolved.notices, resolved.errors

In [ ]:
resolved.events

In [ ]:
resolved.relations

## 3. Requirements: `check_requirements(relations, events)`

Checks RQ1 (no handling unit with different provided locations at the same time), RQ2 ($M \subseteq dom(uloc)$) and RQ3 (no handling unit in two movement events at the same time).
Parameters: the `relations` and `events` frames of step 2.
Returns a list of `BlockingError`s (code, message, affected ids); empty if all requirements hold.
If it is not empty, nothing is derived.

In [ ]:
check_requirements(resolved.relations, resolved.events)

## 4. Current locations, step by step

Each level function takes the relations frame and returns it with one more column.
They build on each other, so they are called in this order.

### Level 0 (provided): `add_loc_0(relations)`

$loc_0((e, hu)) = uloc(e)$ if $e \in dom(uloc)$, else $\bot$; a copy of column `uloc`.

In [ ]:
rel = add_loc_0(resolved.relations)
rel.select("hu", "event", "time", "activity", "uloc", "loc_0")

### Level 1 (forward fill): `add_loc_1(relations)`

$loc_1((e, hu))$ = the provided location of the handling unit's latest located event at or before the time of $e$ ($\le$).

How it works: the relations are collapsed to *time steps* (one row per handling unit and timestamp, holding the first non-null `uloc`; `_per_time_step`), forward filled per handling unit in time order, and joined back.
So all relations of a handling unit at one timestamp get the same $loc_1$.

Example: (c1, e14) has no provided location; the latest located event of c1 before it is e13 with T, so $loc_1 = T$.
(c1, e3) stays $\bot$: c1 has no located event before e5.

In [ ]:
rel = add_loc_1(rel)
rel.select("hu", "event", "time", "activity", "uloc", "loc_0", "loc_1")

### Level 2 (backward fill): `first_location(relations)` and `add_loc_2(relations)`

`first_location` gives, per handling unit, $t_M(hu)$ (time of its first movement, column `t_m`, null if none) and $l_0(hu)$, its location before the first movement:
1. $uloc(e_f)$ of the earliest located event $e_f$, if $e_f$ lies strictly before $t_M(hu)$ (or there is no movement);
2. else the pickup location $puloc$ of the first movement, if provided;
3. else $\bot$.

`add_loc_2` sets $loc_2 = loc_1$ where known, else $l_0(hu)$.
Both take the relations frame (columns hu, time, is_movement, uloc, puloc; `add_loc_2` also loc_1).

Example:
- s1 has no movement; its earliest located event is e2 (A1), so $l_0(s1) = A1$ and (s1, e1) gets A1.
- c2: its earliest located event e6 *is* its first movement (not strictly before), so case 2: $puloc(e6) = A1$, and (c2, e3), (c2, e4) get A1.
- c1: its first located event e5 is its first movement and has no pickup location, so $l_0(c1) = \bot$.

In [ ]:
first_location(rel)

In [ ]:
rel = add_loc_2(rel)
rel.select("hu", "event", "time", "activity", "is_movement", "uloc", "puloc", "loc_1", "loc_2")

### Agreed event locations at Level 2: `event_locations(relations, events, location)`

Level 3 needs the agreed location of an event, $eloc_{loc_2}(e)$.
`event_locations` computes, per event, under the location column `location` (e.g. `"loc_2"`):
- `n_relations`, `n_located`: number of hu-event relations of the event, and of those with a location $\ne \bot$;
- `agreeing`: at most one distinct location $\ne \bot$ among its handling units ($\bot$ does not count against agreement);
- `eloc`: that location if there is exactly one, else $\bot$;
- `located`: fully located / partially located / unlocated.

`events` is passed so that events without handling units are included (as unlocated).

Example: e14 is disagreeing at Level 2 (c1 at T, c3 at B1), so $eloc = \bot$.

In [ ]:
eloc_2 = event_locations(rel, resolved.events, "loc_2")
eloc_2

### Level 3 (transformation locations): `add_loc_3(relations, eloc_2)`

$loc_3 = loc_2$ where known, else $eloc_{loc_2}(t_1)$ for the handling unit's earliest transformation event $t_1(hu)$ before its first movement (or any, if it has none) with $eloc_{loc_2}(t_1) \ne \bot$.
Ties: earliest time, then smallest event id.

| parameter | meaning |
|---|---|
| `relations` | the relations frame with loc_2 (also needs event, is_transformation, is_movement, uloc, puloc) |
| `eloc_2` | columns event, eloc of `event_locations(..., "loc_2")` |

Example:
- c1: $t_1(c1) = e3$ (roll, before $t_M(c1) = 5$) with $eloc_{loc_2}(e3) = A1$, so (c1, e3) and (c1, e4) get A1.
- c4: no movement; $t_1(c4) = e15$ (cut) with $eloc_{loc_2}(e15) = B1$, so both relations of c4 get B1.
- c5 has no located event and no transformation event, so it stays $\bot$ on every level.

In [ ]:
rel = add_loc_3(rel, eloc_2.select("event", "eloc"))
rel.select("hu", "event", "time", "activity", "is_transformation", "loc_0", "loc_1", "loc_2", "loc_3")

## 5. Location before an event and previous location

- `add_before(relations, level)`: $before_{loc_k}((e, hu))$ = $loc_k$ at the handling unit's latest *strictly* earlier time step with $loc_k \ne \bot$ (forward fill shifted by one time step).
- `add_ploc(relations, level)`: $ploc_k = puloc(e)$ where provided, else $before_{loc_k}$.

Parameters: the relations frame (with column loc_k; `add_ploc` also needs puloc and before_k) and the level $k$.
Each call adds one column (`before_k` or `ploc_k`); they are called once per level.

Example (Level 1): (c1, e13) has $before_1 = A2$, but $puloc(e13) = A1$, so $ploc_1 = A1$.

In [ ]:
for level in (0, 1, 2, 3):
    rel = add_ploc(add_before(rel, level), level)
rel.select("hu", "event", "time", "activity", "puloc", "loc_1", "before_1", "ploc_1")

## 6. All in one call: `derive(ocel, config, tables=None)`

`derive` runs steps 2 to 5 (and the flags, see `docs/resources/location-derivation.md`) and returns a `Derivation` with
- `relations`: the relations frame with loc_k, before_k and ploc_k for all levels,
- `events`, `handling_units`, `notices`, `errors`: as in step 2 and 3,
- `event_locations`: dict level → output of `event_locations` at that level,
- `flags`: flags F1-F5 at the selected level,
- `loc`, `ploc`: names of the columns of the selected level (here `loc_3`, `ploc_3`).

Parameters as for `resolve_inputs`.
The result equals the step-by-step frame above.

In [ ]:
derivation = derive(ocel, config)
derivation.loc, derivation.ploc, derivation.relations.equals(rel.select(derivation.relations.columns))

Overview of all levels (the concept's table "Overview of all levels", also in `data/running_example/expected/levels.csv`):

In [ ]:
derivation.relations.select("hu", "event", "time", "activity", "loc_0", "loc_1", "loc_2", "loc_3")

## 7. Save the mapping of each level

Functions in `level_files.py`:

| function | parameters | result |
|---|---|---|
| `level_mapping(derivation, level)` | a `Derivation` without blocking errors; level $k$ (any level, independent of `config.level`) | frame of one level (columns below) |
| `save_level_mappings(derivation, folder, levels=(0, 1, 2, 3))` | target folder (created if missing; files are overwritten); levels to write | the written paths |
| `load_level_mapping(folder, level)` | folder and level | the frame, with the same types as `level_mapping` |

File `locations_level_{k}.csv`, one row per hu-event relation, sorted by hu, time, event, with columns
event, hu, object_type, time, activity, is_movement, is_transformation, `loc` ($= loc_k$), `ploc` ($= ploc_k$).
`loc` and `ploc` carry no level suffix, so all files have the same columns and the level is in the file name.
Time is written in ISO format and $\bot$ as an empty cell, so the files open in any spreadsheet.

Other notebooks load a level with e.g. `load_level_mapping(DERIVED, 3)`.
A test (`test_level_files.py`) checks that the saved files match the current derivation; rerun this notebook after a change to the derivation or the running example.

In [ ]:
save_level_mappings(derivation, DERIVED)

In [ ]:
load_level_mapping(DERIVED, 3)